# Capstone Data Merging: Bridging the Gap - Education, Labour and Wages in Ontario.

This notebook aims to merge the two datasets we've cleaned earlier.

      - Education Level Data: contains the average hourly wages by gender, education, year and immigrant status.
      - CIS Data: This contains the average annual earning, total income and wages by demographic attributes.

1. Import Libraries and Load Data

In [1]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\Foluso\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [3]:
edu_data = pd.read_excel('../pivot_output/education_pivot_output.xlsx')
cis_data = pd.read_excel('../pivot_output/cis_pivot_output.xlsx')
print()
print(edu_data.head())
print()
print(cis_data.head())


   year                             education                       immigrant  \
0  2006                   High school diploma  Born in Canada (non-immigrant)   
1  2006                   High school diploma                       Immigrant   
2  2006                 Less than high school  Born in Canada (non-immigrant)   
3  2006                 Less than high school                       Immigrant   
4  2006  Postsecondary certificate or diploma  Born in Canada (non-immigrant)   

   Avg_value Gender  
0  16.453333   Male  
1  12.237333   Male  
2  14.173333   Male  
3   7.579333   Male  
4  17.970000   Male  

   year                immigrant_status                             education  \
0  2018  Born in Canada (Non-Immigrant)                   High school diploma   
1  2018  Born in Canada (Non-Immigrant)                   High school diploma   
2  2018  Born in Canada (Non-Immigrant)                 Less than high school   
3  2018  Born in Canada (Non-Immigrant)                

2. Renaming the columns so that both datasets have standardized descriptive names.

In [4]:
cis_data.rename(columns={
    "immigrant_status": "immigrant",
    "Average of earnings": "avg_earnings",
    "Average of wages_salary": "avg_wage_salary",
    "Average of total_income": "avg_total_income"
}, inplace=True)

edu_data.rename(columns={
    "Average of Women": "avg_women_wage",
    "Average of Men": "avg_men_wage",
    "Gender": "gender"
}, inplace=True)


3. Cleaning and Standardizing the Text Columns.

In [5]:
def clean_text(text):
    return (
        str(text).strip().lower()
        .replace("/", " or ")
        .replace("–", "-")
        .replace("  ", " ")
    )

for df in [cis_data, edu_data]:
    for col in ['education', 'immigrant', 'gender']:
        df[col] = df[col].astype(str).apply(clean_text)


4. Making sure the category names in the two datasets define education and Immigration in the same way.

In [6]:
edu_map = {
    'less than high school': 'less than high school',
    'high school diploma': 'high school diploma',
    'postsecondary certificate or diploma': 'postsecondary certificate or diploma',
    'university degree': 'university degree'
}
for df in [cis_data, edu_data]:
    df['education'] = df['education'].replace(edu_map)

imm_map = {
    'landed immigrant': 'immigrant',
    'immigrant': 'immigrant',
    'born in canada (non-immigrant)': 'born in canada (non-immigrant)',
    'non-immigrant (born in canada)': 'born in canada (non-immigrant)'
}
for df in [cis_data, edu_data]:
    df['immigrant'] = df['immigrant'].replace(imm_map)


5. Aligning the years Across the datasets to prevent empty merges that could arise from a year being in one dataset and not being in the second.

In [7]:
common_years = sorted(set(cis_data['year']).intersection(set(edu_data['year'])))
cis_data = cis_data[cis_data['year'].isin(common_years)]
edu_data = edu_data[edu_data['year'].isin(common_years)]

6. Merging the two datasets.

In [8]:
merged_data = pd.merge(
    cis_data,
    edu_data,
    on=['year', 'education', 'immigrant', 'gender'],
    how='inner'
)

print("Merged rows:", len(merged_data))
merged_data.head()


Merged rows: 80


,year,immigrant,education,gender,avg_earnings,avg_wage_salary,avg_total_income,Avg_value
0,2018,born in canada (non-immigrant),high school diploma,female,17503.608247,16681.857911,31596.089198,17.913333
1,2018,born in canada (non-immigrant),high school diploma,male,31877.938537,30441.083445,44415.982503,19.660000
2,2018,born in canada (non-immigrant),less than high school,female,6052.435897,5907.230769,21835.102564,12.563333
3,2018,born in canada (non-immigrant),less than high school,male,15215.602022,14607.238051,31381.617647,16.133333
4,2018,born in canada (non-immigrant),postsecondary certificate or diploma,female,30015.391288,28700.144564,44496.752120,19.630000


7. Checking for nulls

In [9]:
merged_data.isnull().sum()

year                0
immigrant           0
education           0
gender              0
avg_earnings        0
avg_wage_salary     0
avg_total_income    0
Avg_value           0
dtype: int64

8. Exporting the Final Merged Dataset.

In [10]:
merged_data.to_excel("merged_final.xlsx", index=False)
print("File exported successfully.")


File exported successfully.
